# Infer-16-Sparse-Gaussian-Process : Processus Gaussiens et frontières non-linéaires

**Série** : Programmation Probabiliste avec Infer.NET (23)
**Durée estimée** : 55 minutes
**Prérequis** : [Infer-9-Classification](Infer-9-Classification.ipynb) (Bayes Point Machine), [Infer-2-Gaussian-Mixtures](Infer-2-Gaussian-Mixtures.ipynb), [Infer-12-Modeles-Hierarchiques](Infer-12-Modeles-Hierarchiques.ipynb) (comprendre pourquoi un prior partagé est plus robuste qu'un prior isolé)

---

## Objectifs

- Comprendre le **processus gaussien** comme une distribution sur des fonctions
- Construire un prior via un **noyau** (kernel RBF / squared-exponential)
- Implémenter la **classification GP** (modèle probit) sur des données non linéairement séparables
- Mesurer le rôle du **basis set** (points inducteurs) dans l'approximation sparse
- Contraster avec le Bayes Point Machine (Infer-9) : frontière linéaire vs non linéaire

---

## Navigation

| Précédent | Suivant |
|-----------|--------|
| [Infer-15-Recommenders](Infer-15-Recommenders.ipynb) | [Infer-17-Kalman-Filter](Infer-17-Kalman-Filter.ipynb) |

---

## 1. Configuration

Cette section prepare l'environnement Infer.NET. Le processus gaussien necessite les espaces de noms `.Distributions`, `.Distributions.Kernels` (noyaux) et `.Math` (Vector).

In [1]:
#r "nuget: Microsoft.ML.Probabilistic"
#r "nuget: Microsoft.ML.Probabilistic.Compiler"

using Microsoft.ML.Probabilistic;
using Microsoft.ML.Probabilistic.Distributions;
using Microsoft.ML.Probabilistic.Distributions.Kernels;
using Microsoft.ML.Probabilistic.Math;
using Microsoft.ML.Probabilistic.Models;
using Microsoft.ML.Probabilistic.Algorithms;
using Microsoft.ML.Probabilistic.Compiler;

Console.WriteLine("Infer.NET pret pour les processus gaussiens.");

Installing Packages Microsoft.ML.Probabilistic Microsoft.ML.Probabilistic.Compiler

Infer.NET pret pour les processus gaussiens.


Chargement du helper de visualisation des graphes de facteurs (reutilise dans toute la serie).

In [2]:
#load "FactorGraphHelper.cs"

if (FactorGraphHelper.IsGraphvizAvailable())
    Console.WriteLine("Graphviz disponible - les graphes de facteurs seront affiches automatiquement.");
else
    Console.WriteLine("Graphviz non installe - utilisez viz-js.com pour visualiser les fichiers .gv generes.");

Graphviz disponible - les graphes de facteurs seront affiches automatiquement.


## 2. Motivation : au-dela de la frontiere lineaire

[Infer-9](Infer-9-Classification.ipynb) introduisait le **Bayes Point Machine** : un classifieur qui marginalise sur tous les hyperplans separateurs. Mais un hyperplan est **lineaire** dans l'espace des features. Des qu'il faut une frontiere **courbe**, le BPM est impuissant.

**Exemple canonique : le "donut".** Deux classes organisees en cercles concentriques -- la classe positive forme un anneau exterieur, la classe negative un disque interieur. Aucune droite (aucun hyperplan) ne peut separer ces deux classes, parce que chaque demi-plan coupe forcement l'une et l'autre.

| Approche | Frontiere | Donut |
|----------|-----------|-------|
| BPM (Infer-9) | Hyperplan | **Echec** (lineaire) |
| GP (ce notebook) | Courbe quelconque | **Succes** (noyau RBF) |

Le processus gaussien resout ce probleme en placant un **prior sur des fonctions** : au lieu d'inferer un vecteur de poids $\mathbf{w}$ (qui definit un hyperplan), on infere une **fonction** $f$ tireee d'un processus gaussien, puis on classifie via un modele probit $y = \mathbb{1}[f(\mathbf{x}) > 0]$.

## 3. Un prior sur des fonctions

Un **processus gaussien** est defini par deux objets :

- une **fonction moyenne** $m(\mathbf{x})$ (souvent constante, ici nulle) ;
- un **noyau de covariance** $k(\mathbf{x}, \mathbf{x}')$ qui dit a quel point deux points sont correlles.

Le noyau **squared-exponential** (RBF) decroche avec la distance :

$$k(\mathbf{x}, \mathbf{x}') = \exp\left(-\frac{\|\mathbf{x}-\mathbf{x}'\|^2}{2\ell^2}\right)$$

ou $\ell$ est la **longueur de correlation** : deux points proches ($\|\mathbf{x}-\mathbf{x}'\| \ll \ell$) ont des valeurs de $f$ quasi-egales ; deux points eloignes ($\gg \ell$) sont decorreles. C'est cette coherence locale qui produit des frontieres **lisses** plutot que du bruit.

In [3]:
// Le prior gaussien sur les fonctions : moyenne nulle + noyau RBF.
GaussianProcess gpPrior = new GaussianProcess(
    new ConstantFunction(0),      // moyenne m(x) = 0
    new SquaredExponential(0));   // noyau RBF k(x,x') = exp(-||x-x'||^2 / 2)

Console.WriteLine("Prior GP defini.");
Console.WriteLine($"  Moyenne : ConstantFunction(0)");
Console.WriteLine($"  Noyau  : SquaredExponential (RBF)");

// Un noyau mesure la similitude entre deux points. La table ci-dessous montre
// la covariance RBF entre quelques points du plan -- proche de 1 quand les
// points coincident, decroissant vers 0 avec la distance.
Vector[] probes = {
    Vector.FromArray(0.0, 0.0),
    Vector.FromArray(0.5, 0.0),
    Vector.FromArray(1.0, 0.0),
    Vector.FromArray(2.0, 0.0)
};
Console.WriteLine("\nCovariance RBF depuis l'origine (0,0) :");
foreach (var p in probes)
{
    double d = Math.Sqrt(p[0]*p[0] + p[1]*p[1]);
    double cov = Math.Exp(-0.5 * d * d);
    Console.WriteLine($"  vers {p} (distance {d:F2}) : k = {cov:F4}");
}

Prior GP defini.


  Moyenne : ConstantFunction(0)


  Noyau  : SquaredExponential (RBF)



Covariance RBF depuis l'origine (0,0) :


  vers 0 0 (distance 0,00) : k = 1,0000


  vers 0,5 0 (distance 0,50) : k = 0,8825


  vers 1 0 (distance 1,00) : k = 0,6065


  vers 2 0 (distance 2,00) : k = 0,1353


**Lecture** : la covariance decroit exponentiellement avec la distance. Un point eloigne de 2 unites est deja quasi-decorrele de l'origine -- ses valeurs de $f$ sont donc independantes. Cette decroissance est ce qui rend le GP **local** : la frontiere ne peut pas osciller sauvagement, elle est contrainte par la coherence que le noyau impose.

## 4. Le dataset "donut" (non lineairement separable)

Construisons les donnees : un disque interieur (classe negative) entoure d'un anneau (classe positive).

In [4]:
// Dataset "donut" : disque interieur (false) + anneau exterieur (true).
// Aucun hyperplan ne separe ces deux classes.
var rnd = new System.Random(7);
var inputsList = new System.Collections.Generic.List<Vector>();
var outputsList = new System.Collections.Generic.List<bool>();

// Disque interieur : rayon ~ 0.3-0.6  => classe false
for (int i = 0; i < 8; i++)
{
    double r = 0.30 + 0.30 * rnd.NextDouble();
    double t = 2.0 * Math.PI * rnd.NextDouble();
    inputsList.Add(Vector.FromArray(r * Math.Cos(t), r * Math.Sin(t)));
    outputsList.Add(false);
}
// Anneau exterieur : rayon ~ 1.2-1.5 => classe true
for (int i = 0; i < 8; i++)
{
    double r = 1.20 + 0.30 * rnd.NextDouble();
    double t = 2.0 * Math.PI * rnd.NextDouble();
    inputsList.Add(Vector.FromArray(r * Math.Cos(t), r * Math.Sin(t)));
    outputsList.Add(true);
}

Vector[] inputsDonut = inputsList.ToArray();
bool[] outputsDonut = outputsList.ToArray();

Console.WriteLine($"Dataset donut : {inputsDonut.Length} points");
Console.WriteLine($"  Classe false (disque interieur) : {outputsDonut.Count(b => !b)} points");
Console.WriteLine($"  Classe true  (anneau exterieur) : {outputsDonut.Count(b => b)} points");
Console.WriteLine("\nPremiers points :");
for (int i = 0; i < 4; i++)
    Console.WriteLine($"  x{i} = {inputsDonut[i]} -> classe {outputsDonut[i]}");

Dataset donut : 16 points


  Classe false (disque interieur) : 8 points


  Classe true  (anneau exterieur) : 8 points



Premiers points :


  x0 = 0,2864 -0,3002 -> classe False


  x1 = 0,4717 0,1607 -> classe False


  x2 = -0,1834 -0,3666 -> classe False


  x3 = 0,2997 -0,08919 -> classe False


### Pourquoi le BPM echouerait ici

Un BPM (Infer-9) cherche un hyperplan $\mathbf{w}^T\mathbf{x} - b = 0$. Quelle que soit la direction $\mathbf{w}$, la demi-droite coupe a la fois le disque interieur et l'anneau exterieur : il classera forcement faux une partie des points. La non-separabilite lineaire est totale. C'est precisement le cas ou un GP excelle.

## 5. Le modele : classification GP (probit)

On assemble le modele en suivant le meme schelet probit qu'Infer-9, mais en remplacant le score lineaire $\mathbf{w}^T\mathbf{x}$ par une **fonction** $f(\mathbf{x})$ tiree du processus gaussien :

$$y_j = \mathbb{1}[f(\mathbf{x}_j) + \epsilon_j > 0], \qquad \epsilon_j \sim \mathcal{N}(0, 0.1)$$

ou $f$ est un GP sparse (approxime sur un **basis set** de points induiseurs).

In [5]:
// Modele : classification GP probit sur le donut.

// 1. Prior sparse sur la fonction f.
Variable<SparseGP> prior = Variable.New<SparseGP>().Named("prior");
Variable<IFunction> f = Variable<IFunction>.Random(prior).Named("f");

// 2. Observations.
VariableArray<Vector> x = Variable.Observed(inputsDonut).Named("x");
Range j = x.Range.Named("j");
VariableArray<bool> y = Variable.Observed(outputsDonut, j).Named("y");

// 3. Modele probit : score = f(x_j), bruite gaussien, seuil a 0.
Variable<double> score = Variable.FunctionEvaluate(f, x[j]);
y[j] = (Variable.GaussianFromMeanAndVariance(score, 0.1) > 0);

// 4. Prior GP + basis (grille de points induiseurs couvrant le plan).
GaussianProcess gp = new GaussianProcess(new ConstantFunction(0), new SquaredExponential(0));
Vector[] basis = new Vector[] {
    Vector.FromArray(-1.5, -1.5), Vector.FromArray(0.0, -1.5), Vector.FromArray(1.5, -1.5),
    Vector.FromArray(-1.5,  0.0), Vector.FromArray(0.0,  0.0), Vector.FromArray(1.5,  0.0),
    Vector.FromArray(-1.5,  1.5), Vector.FromArray(0.0,  1.5), Vector.FromArray(1.5,  1.5)
};
prior.ObservedValue = new SparseGP(new SparseGPFixed(gp, basis));

// 5. Inference EP.
InferenceEngine engine = new InferenceEngine(new ExpectationPropagation());
engine.Compiler.CompilerChoice = CompilerChoice.Roslyn;
engine.ShowProgress = false;
engine.ShowFactorGraph = true;

SparseGP sgp = engine.Infer<SparseGP>(f);
Console.WriteLine("Posterior SparseGP inferer.");
Console.WriteLine($"  Basis set : {basis.Length} points induiseurs");

Posterior SparseGP inferer.


  Basis set : 9 points induiseurs


In [6]:
// Visualisation du graphe de facteurs du classifieur GP.
display(HTML(FactorGraphHelper.GetLatestFactorGraphHtml()));

Model_07_01_26_03_15_19_62.svg 
 
 <?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 14.1.2 (20260124.0452)
 -->
<!-- Title: Model Pages: 1 -->
 
 
 Model 
 
<!-- node0 -->
 
 node0 
 
 prior 
 
<!-- node1 -->
 
 node1 
 
 Random 
 
<!-- node0->node1 -->
 
 node0->node1 
 
 
 dist 
 
<!-- node2 -->
 
 node2 
 
 f 
 
<!-- node1->node2 -->
 
 node1->node2 
 
 
 
<!-- node3 -->
 
 node3 
 
 FunctionEvaluate 
 
<!-- node2->node3 -->
 
 node2->node3 
 
 
 func 
 
<!-- node5 -->
 
 node5 
 
 vdouble[]0[j] 
 
<!-- node3->node5 -->
 
 node3->node5 
 
 
 
<!-- node4 -->
 
 node4 
 
 x[j] 
 
<!-- node4->node3 -->
 
 node4->node3 
 
 
 x 
 
<!-- node6 -->
 
 node6 
 
 GaussianFromMeanAndVariance 
 
<!-- node5->node6 -->
 
 node5->node6 
 
 
 mean 
 
<!-- node8 -->
 
 node8 
 
 vdouble[]1[j] 
 
<!-- node6->node8 -->
 
 node6->node8 
 
 
 
<!-- node7 -->
 
 node7 
 
 0,1 
 
<!-- node7->node6 -->
 
 node7->node6 
 
 
 variance 
 
<!-- node9 -->
 
 node9 
 
 IsPositive 
 
<!-- node8->node9 -->
 
 node8->node9 
 
 
 x 
 
<!-- node10 -->
 
 node10 
 
 y[j] 
 
<!-- node9->node10 -->
 
 node9->node10


warning CS1701: En supposant que la référence d'assembly 'Microsoft.AspNetCore.Html.Abstractions, Version=2.3.0.0, Culture=neutral, PublicKeyToken=adb9793829ddae60' utilisée par 'Microsoft.DotNet.Interactive' correspond à l'identité 'Microsoft.AspNetCore.Html.Abstractions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=adb9793829ddae60' de 'Microsoft.AspNetCore.Html.Abstractions', il se peut que vous deviez fournir une stratégie runtime



### Lecture du graphe

Le graphe de facteurs montre la structure : la variable $f$ (une fonction, pas un scalaire) est connectee a chaque observation via le facteur `FunctionEvaluate`. Le prior `SparseGP` repose $f$ sur le basis de 9 points. L'EP propage les messages a travers cette chaine pour produire le posterior sur $f$.

## 6. Predictions avec incertitude

Le posterior `SparseGP` permet d'evaluer $f$ en tout nouveau point via `Marginal`. On peut ainsi tracer la **frontiere de decision** (la ou $f$ change de signe) et la **confiance** (distance a 0).

In [7]:
// Predictions sur le training set + quelques points test.
Console.WriteLine("=== Predictions sur le training set ===");
int correct = 0;
for (int i = 0; i < outputsDonut.Length; i++)
{
    Gaussian post = sgp.Marginal(inputsDonut[i]);
    double postMean = post.GetMean();
    bool pred = postMean > 0.0;
    if (pred == outputsDonut[i]) correct++;
    Console.WriteLine($"  x{i} {inputsDonut[i],20} : f={post,30} pred={pred} reel={outputsDonut[i]}");
}
Console.WriteLine($"\nExactitude training : {correct}/{outputsDonut.Length} = {100.0*correct/outputsDonut.Length:F1}%");

=== Predictions sur le training set ===


  x0       0,2864 -0,3002 : f=     Gaussian(-0,7629, 0,1817) pred=False reel=False


  x1        0,4717 0,1607 : f=     Gaussian(-0,6213, 0,1934) pred=False reel=False


  x2      -0,1834 -0,3666 : f=      Gaussian(-0,712, 0,1868) pred=False reel=False


  x3      0,2997 -0,08919 : f=     Gaussian(-0,8189, 0,1491) pred=False reel=False


...

  x12         1,295 0,2349 : f=      Gaussian(0,4897, 0,1892) pred=True reel=True


  x13         1,468 0,2313 : f=       Gaussian(0,657, 0,1844) pred=True reel=True


  x14       -0,2632 -1,318 : f=      Gaussian(0,3817, 0,3108) pred=True reel=True


  x15        -0,4181 1,369 : f=       Gaussian(0,5901, 0,345) pred=True reel=True



Exactitude training : 16/16 = 100,0%


In [8]:
// Test sur des points nouveaux : au centre (doit etre false), a mi-rayon (incertain),
// sur l'anneau (doit etre true), loin (true).
Vector[] testPoints = {
    Vector.FromArray(0.0, 0.0),     // centre du disque -> false attendu
    Vector.FromArray(0.9, 0.0),     // entre les deux -> zone d'incertitude
    Vector.FromArray(1.3, 0.0),     // anneau -> true attendu
    Vector.FromArray(0.0, 1.4)      // anneau (haut) -> true attendu
};
Console.WriteLine("=== Predictions sur points test ===");
foreach (var tp in testPoints)
{
    Gaussian post = sgp.Marginal(tp);
    double p = MMath.NormalCdf(post.GetMean() / Math.Sqrt(post.GetVariance() + 0.1));
    Console.WriteLine($"  {tp,16} : E[f]={post.GetMean():+0.000;-0.000} P(classe=true)={p:F3}");
}

=== Predictions sur points test ===


               0 0 : E[f]=-0,910 P(classe=true)=0,026


             0,9 0 : E[f]=-0,092 P(classe=true)=0,435


             1,3 0 : E[f]=+0,449 P(classe=true)=0,801


             0 1,4 : E[f]=+0,507 P(classe=true)=0,769


### Analyse

- Le **centre** du donut est correctement classe `false` ($E[f] < 0$) bien qu'aucune donnee ne soit exactement la -- la coherence du noyau propage la classe du disque interieur vers le centre.
- La **zone d'incertitude** (mi-rayon, entre le disque et l'anneau) montre une probabilite proche de 0.5 : le GP sait qu'il ne sait pas.
- L'**anneau** est correctement `true`.

C'est le comportement que le BPM (lineaire) ne peut pas reproduire : il n'y a pas d'hyperplan qui vaut `false` au centre et `true` tout autour. Le GP, lui, a appris une frontiere **circulaire**.

## 7. Sparse : le role du basis set

Un GP "full" evalue la fonction sur **tous** les points d'entrainement, ce qui coute $O(n^3)$. L'approximation **sparse** remplace ce jeu complet par un petit **basis set** de $m$ points induiseurs : le cout tombe a $O(nm^2)$, controlable par le choix de $m$.

> La doc Infer.NET precise : *« If the basis set is exactly the set of inputs, then the distribution is equivalent to a full (non-sparse) Gaussian Process. »* Le sparse est donc un vrai continu, pas une variante degradee -- avec $m = n$ on retrouve le GP exact.

Comparons deux tailles de basis : peu d'inducteurs (sparse agressif) vs basis = tous les inputs (full).

In [9]:
// Comparaison : basis sparse (4 points) vs basis = tous les inputs (full).
Vector[] basisSparse = new Vector[] {
    Vector.FromArray(0.0, 0.0), Vector.FromArray(1.3, 0.0),
    Vector.FromArray(-1.3, 0.0), Vector.FromArray(0.0, 1.3)
};

GaussianProcess gp2 = new GaussianProcess(new ConstantFunction(0), new SquaredExponential(0));

// (a) Sparse avec peu de points induiseurs.
Variable<SparseGP> priorSparse = Variable.New<SparseGP>().Named("priorSparse");
Variable<IFunction> fSparse = Variable<IFunction>.Random(priorSparse).Named("fSparse");
VariableArray<Vector> xS = Variable.Observed(inputsDonut).Named("xS");
Range jS = xS.Range.Named("jS");
VariableArray<bool> yS = Variable.Observed(outputsDonut, jS).Named("yS");
Variable<double> scoreS = Variable.FunctionEvaluate(fSparse, xS[jS]);
yS[jS] = (Variable.GaussianFromMeanAndVariance(scoreS, 0.1) > 0);
priorSparse.ObservedValue = new SparseGP(new SparseGPFixed(gp2, basisSparse));

var engSparse = new InferenceEngine(new ExpectationPropagation());
engSparse.Compiler.CompilerChoice = CompilerChoice.Roslyn;
engSparse.ShowProgress = false;
SparseGP sgpSparse = engSparse.Infer<SparseGP>(fSparse);

// (b) "Full" : basis = tous les inputs (n = 16).
Variable<SparseGP> priorFull = Variable.New<SparseGP>().Named("priorFull");
Variable<IFunction> fFull = Variable<IFunction>.Random(priorFull).Named("fFull");
VariableArray<Vector> xF = Variable.Observed(inputsDonut).Named("xF");
Range jF = xF.Range.Named("jF");
VariableArray<bool> yF = Variable.Observed(outputsDonut, jF).Named("yF");
Variable<double> scoreF = Variable.FunctionEvaluate(fFull, xF[jF]);
yF[jF] = (Variable.GaussianFromMeanAndVariance(scoreF, 0.1) > 0);
priorFull.ObservedValue = new SparseGP(new SparseGPFixed(gp2, inputsDonut));

var engFull = new InferenceEngine(new ExpectationPropagation());
engFull.Compiler.CompilerChoice = CompilerChoice.Roslyn;
engFull.ShowProgress = false;
SparseGP sgpFull = engFull.Infer<SparseGP>(fFull);

// Exactitude des deux.
int EvalAcc(SparseGP sgp, Vector[] xs, bool[] ys)
{
    int c = 0;
    for (int i = 0; i < ys.Length; i++)
        if ((sgp.Marginal(xs[i]).GetMean() > 0) == ys[i]) c++;
    return c;
}
Console.WriteLine("=== Sparse vs Full ===");
Console.WriteLine($"  Sparse (basis=4 induiseurs)  : {EvalAcc(sgpSparse, inputsDonut, outputsDonut)}/{outputsDonut.Length}");
Console.WriteLine($"  Full   (basis=16=inputs)     : {EvalAcc(sgpFull, inputsDonut, outputsDonut)}/{outputsDonut.Length}");
Console.WriteLine("\nLes deux separnt le donut -- le sparse economise du calcul sans casser la separabilite.");

=== Sparse vs Full ===


  Sparse (basis=4 induiseurs)  : 15/16


  Full   (basis=16=inputs)     : 16/16



Les deux separnt le donut -- le sparse economise du calcul sans casser la separabilite.


### Lecture : pourquoi le sparse marche ici

Le donut est une structure **globale** (un cercle), capturable meme par peu de points induiseurs bien places. Le sparse est le plus rentable quand le signal est lisse a grande echelle (peu de degres de liberte effectifs) ; il perd en precision quand la frontiere oscille finement. Le basis set est un **biais-variance** : trop peu d'inducteurs sous-ajustent, trop = cout full sans gain.

| Basis | Cout | Risque |
|-------|------|--------|
| Petit ($m \ll n$) | $O(nm^2)$ economique | Sous-ajustement si frontiere fine |
| $m = n$ (full) | $O(n^3)$ | Exact mais cher

## 8. Exercices

### Exercice 1 : Longueur de correlation

Le noyau `SquaredExponential(0)` utilise une longueur de correlation par defaut. Observez comment une **longueur trop courte** (frontiere herissee, surajustement) ou **trop longue** (frontiere trop lisse, sous-ajustement) degrade la classification.

In [10]:
// Exercice : influer sur la longueur de correlation du noyau.
// Indice : le constructeur SquaredExponential accepte des parametres de scale.
// Essayez plusieurs valeurs et mesurez l'exactitude training.
// Console.WriteLine($"length=... : {acc}/{n}");
Console.WriteLine("Exercice a completer : longueur de correlation du noyau");

Exercice a completer : longueur de correlation du noyau


### Exercice 2 : Donut bruite

Ajoutez du bruit aux etiquettes (quelques points du disque etiquetes `true`, et reciproquement). Le GP gere-t-il le bruit mieux qu'un classifieur deterministe ? Quantifiez la **confiance** (P proche de 0.5) sur les points contredits.

In [11]:
// Exercice : flippez 2 etiquettes et re-inferez. Observez P(classe) sur les points bruites.
// Indice : construire un nouveau tableau outputsNoisy avec 2 valeurs inversee.
Console.WriteLine("Exercice a completer : robustesse au bruit du GP");

Exercice a completer : robustesse au bruit du GP


### Exercice 3 : Donnees lineairement separables

Construisez un dataset **lineairement** separable (deux amas de part et d'autre d'une droite). Comparez le GP au BPM (reutilisez `SimpleBPM` d'Infer-9) : sur un probleme simple, le GP est-il utile, ou le BPM suffit-il ?

In [12]:
// Exercice : dataset lineairement separable -> GP vs BPM (Infer-9).
// Indice : deux amas centres en (-1,-1) et (+1,+1).
// Question : sur un cas lineaire, le GP apporte-t-il plus que le BPM ?
Console.WriteLine("Exercice a completer : GP vs BPM sur cas lineaire");

Exercice a completer : GP vs BPM sur cas lineaire


### GP vs BPM (Infer-9)

| Aspect | BPM (Infer-9) | GP (Infer-16) |
|--------|---------------|---------------|
| Paramètre | Vecteur de poids $\mathbf{w}$ | Fonction $f$ |
| Frontière | Hyperplan (linéaire) | Courbe quelconque |
| Donut | **Échec** | **Succès** |
| Coût | Linéaire en $n$ | $O(nm^2)$ (sparse) |
| Quand l'utiliser | Données linéairement séparables | Frontières non linéaires |

### Distributions utilisées

| Distribution | Rôle |
|--------------|------|
| `SparseGP` | Prior et posterior sur la fonction $f$ |
| `Gaussian` | Score bruité (probit) + marginal posterior en chaque point |
| `Bernoulli` | (Implicite) prediction de classe via `NormalCdf` |

> **Leçon** : le GP est l'outil naturel quand la frontière de décision est non linéaire ET qu'on veut une **incertitude calibrée**. Pour un cas linéaire, le BPM (Infer-9) reste plus simple et plus rapide. Le processus gaussien n'est pas un remplacement universel — c'est un **complément** qui étend la boîte à outils bayésienne au non-linéaire, au prix d'un noyau à choisir et d'un basis à calibrer.

### Ponts dans la série

- **Pont hiérarchique** ([Infer-12-Modeles-Hierarchiques](Infer-12-Modeles-Hierarchiques.ipynb)) : la longueur d'échelle $\ell$ du noyau RBF est un hyperparamètre ; un GP qui apprendrait une longueur par groupe d'inputs est, structurellement, un modèle hiérarchique sur ces hyperparamètres. Ce notebook, en restant à $\ell$ fixée, montre d'abord le mécanisme bayésien avant d'envisager sa généralisation hiérarchique.
- **Suivant** ([Infer-17-Kalman-Filter](Infer-17-Kalman-Filter.ipynb)) : le GP est le pendant **statique** (covariance dans l'espace des entrées) du modèle d'état markovien (covariance dans le temps). Tous deux sont résolus exactement par EP dans la famille gaussienne — Kalman est le GP temporel, en quelque sorte.